Simple U-NET Implementation

In [35]:
from enum import Enum
from pathlib import Path
from PIL import Image


import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from tqdm import trange

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [37]:
class PillDataset(Dataset):

    def __init__(self, root_dir='datasets/pill'):
        super().__init__()
        self.root_dir = Path(root_dir)
        self.img_path = sorted(
            (self.root_dir / 'train' / 'good').glob('*.png')
        )

        self.transform = transforms.Compose([
            transforms.Resize((800, 800)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]
            )
        ]) 

    def __len__(self):
        return len(self.img_path)

    def __getitem__(self, index):
        path = self.img_path[index]

        image = Image.open(path).convert('RGB')
        image = self.transform(image)

        return image

Hyperparameters

In [38]:
lr = 1e-4
rgb = 3



MVTec AD Dataset

metal_nut = 700 x 700
pill = 800 x 800

In [39]:
metal_nut = 700
pill = 800

In [ ]:

# using pill from MVTec AD dataset

class SimpleUNET (nn.Module):

    def __init__(self):
        super().__init__()

        self.norm1 = nn.GroupNorm(24, 24)
        self.norm2 = nn.GroupNorm(32, 64)
        self.norm3 = nn.GroupNorm(32, 192)


        # (batch x 3 x 800 x 800)
        self.downsamp1 = nn.Conv2d(
            in_channels=rgb, 
            out_channels=24,
            kernel_size=5, 
            padding=0, 
            stride=2,)
        # (batch x 24 x 398 x 398)
        self.downsamp2 = nn.Conv2d(
            in_channels=24,
            out_channels=64, 
            kernel_size=3, 
            padding=0, 
            stride=2)
        # (batch x 64 x 198 x 198)
        self.downsamp3 = nn.Conv2d(
            in_channels=64, 
            out_channels=128, 
            kernel_size=3, 
            padding=0, 
            stride=2)
        # (batch x 128 x 98 x 98)

        self.proj1 = nn.Conv2d(
            in_channels=128,
            out_channels=64,
            kernel_size=3,
            padding=1,
            stride=1
        )

        self.proj2 = nn.Conv2d(
            in_channels=48,
            out_channels=24,
            kernel_size=3,
            padding=1,
            stride=1
        )

        self.time_emb1 = nn.Linear(1, 24)
        # (1 x 24)
        self.time_emb2 = nn.Linear(1, 64)
        # (1 x 64)
        self.time_emb3 = nn.Linear(1, 128)
        # (1 x 128)

        self.bottleneck = nn.Sequential(
            nn.GroupNorm(32, 128),
            nn.SiLU(),
            # (batch x 128 x 98 x 98)
            nn.Conv2d(
                in_channels=128, 
                out_channels=128, 
                kernel_size=3, 
                padding=1,
                stride=1),
            nn.GroupNorm(32, 128),
            nn.SiLU(),
            # (batch x 128 x 98 x 98)
            nn.Conv2d(
                in_channels=128, 
                out_channels=128, 
                kernel_size=3, 
                padding=1,
                stride=1)
        )

        # (batch x 128 x 98 x 98)
        self.upsamp4 = nn.ConvTranspose2d(
            in_channels=128, 
            out_channels=64, 
            kernel_size=3, 
            padding=0, 
            output_padding=1,
            stride=2)
        # (batch x 64 x 198 x 198)
        self.upsamp5 = nn.ConvTranspose2d(
            in_channels=64, 
            out_channels=24, 
            kernel_size=3, 
            padding=0,
            output_padding=1,
            stride=2)
        # (batch x 24 x 398 x 398)
        self.upsamp6 = nn.ConvTranspose2d(
            in_channels=24, 
            out_channels=rgb, 
            kernel_size=5, 
            padding=0, 
            output_padding=1,
            stride=2)
        # (batch x 3 x 800 x 800)


    def forward(self, x: torch.Tensor, t: torch.Tensor):

        # t = t.float()[:, None]
        # # (batch x 1)

        # down-sampling block
        # input
        # (batch x 3 x 800 x 800)


        h1 = self.downsamp1(x)
        # (batch x 24 x 398 x 398)
        h1 = self.norm1(h1)
        h1 = F.silu(h1)
        
        t1 = self.time_emb1(t)
        t1 = t1[:, :, None, None]
        # (batch x 24 x 1 x 1)
        # time-step conditioning
        h1 = h1 + t1

        h2 = self.downsamp2(h1)
        # (batch x 64 x 198 x 198)
        h2 = self.norm2(h2)
        h2 = F.silu(h2)
        
        t2 = self.time_emb2(t)
        t2 = t2[:, :, None, None]
        # (batch x 64 x 1 x 1)
        # time-step conditioning
        h2 = h2 + t2

        h3 = self.downsamp3(h2)
        # (batch x 128 x 98 x 98)
        h3 = self.norm3(h3)
        h3 = F.silu(h3)
        
        t3 = self.time_emb3(t)
        t3 = t3[:, :, None, None]
        # (batch x 128 x 1 x 1)
        # time-step conditioning
        h3 = h3 + t3

        # bottleneck

        # (batch x 128 x 98 x 98)
        latent = self.bottleneck(h3)
        # (batch x 128 x 98 x 98)


        # up-sampling block

        h4 = self.norm3(latent)
        h4 = F.silu(h4)
        h4 = self.upsamp4(h4)
        # (batch x 64 x 198 x 198)
        # time-step conditioning
        h4 = h4 + t2
        # skip connection
        # h4 and h2 have the same shape
        h4 = torch.concat([h4, h2], dim=1)
        # after skip (batch x 128 x 198 x 198)
        h4 = self.proj1(h4)
        # (batch x 64 x 198 x 198)

        h5 = self.norm2(h4)
        h5 = F.silu(h5)
        h5 = self.upsamp5(h5)
        # (batch x 24 x 398 x 398)
        # time-step conditioning
        h5 = h5 + t1
        # skip connection
        # h5 and h1 have the same shape
        h5 = torch.concat([h5, h1], dim=1)
        # after skip (batch x 48 x 398 x 398)
        h5 = self.proj2(h5)
        # (batch x 8 x 398 x 398)

        h6 = self.norm1(h5)
        h6 = F.silu(h6)
        h6 = self.upsamp6(h6)
        # (batch x 3 x 800 x 800)



        # predicted noise
        return h6





In [41]:
class Padding(Enum):
    valid = 0
    same = 1

def sample_size(input: int, kernel:int, padding: Padding=Padding.valid, stride:int=1):

    if not kernel % 2:
        raise ValueError('an odd number is preferred')

    pad = 0 if padding is Padding.valid else (kernel - 1) // 2

    return ((input + (2 * pad) - kernel) // stride) + 1

In [42]:
sample_size(input=800, kernel=5, padding=Padding.valid, stride=2)

398

Training

$x_{t} = \sqrt{\bar{\alpha}}x_{0} + \sqrt{1 - \bar{\alpha_{t}}}\epsilon$

In [43]:
def train(T, epochs, batch=4):

    dataset = PillDataset()

    train_loader = DataLoader(
        dataset,
        batch_size=batch,
        shuffle=True
    )

    beta = torch.linspace(
        1e-4, 
        0.02, # 2e-2
        T,
        device=device)


    alpha = 1 - beta

    alpha_bar = torch.cumprod(alpha, dim=0)
    # [T]

    model = SimpleUNET()
    model = model.to(device)

    optimizer = optim.Adam(
        params=model.parameters(),
        lr=lr
    )

    model.train() 

    for epoch in trange(epochs):

        running_loss = 0.0

        for x0 in train_loader:
            x0 = x0.to(device)
            # current batch size
            b = x0.shape[0]
            # random timestep
            ts = torch.randint(
                low=0,
                high=T,
                size=(b,),
                device=device)
            # [b]
            alpha_bar_t = alpha_bar[ts]
            # [T] --> [b]
            alpha_bar_t = alpha_bar_t[:, None, None, None]
            # [b] --> [b x 1 x 1 x 1]
            # noise
            epsilon = torch.randn_like(x0)
            # noise data
            xt = (torch.sqrt(alpha_bar_t) * x0) + (torch.sqrt(1 - alpha_bar_t) * epsilon)
            # noise prediction
            # normalized 
            tmodel = ts.float()[:, None] / (T - 1)
            epsilon_pred = model(xt, tmodel)

            loss = F.mse_loss(epsilon_pred, epsilon)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        average_loss = running_loss / len(train_loader)

        print(
            f'Epoch {epoch + 1} / {epochs}',
            f'Loss: {average_loss:.6f}'
        )


    return model


In [46]:
epochs = 60

T = 40

model = train(T, epochs, batch=4)

torch.save(
    model.state_dict(),
    "simple_unet-2.pth"
)

  2%|▏         | 1/60 [00:13<13:04, 13.30s/it]

Epoch 1 / 60 Loss: 1.046119


  3%|▎         | 2/60 [00:26<12:50, 13.28s/it]

Epoch 2 / 60 Loss: 0.911553


  5%|▌         | 3/60 [00:39<12:40, 13.34s/it]

...

Epoch 58 / 60 Loss: 0.117803


 98%|█████████▊| 59/60 [13:23<00:13, 13.68s/it]

Epoch 59 / 60 Loss: 0.112129


100%|██████████| 60/60 [13:37<00:00, 13.62s/it]

Epoch 60 / 60 Loss: 0.113589


Test block

In [45]:
dataset = PillDataset()

train_loader = DataLoader(
        dataset,
        batch_size=batch,
        shuffle=True
    )


n = len(train_loader)

T = 25

ts = torch.randint(
    low=0,
    high=T,
    size=(batch,)
    # device=device
)

print('the shape of ts: ', ts.shape)

beta = torch.linspace(
    0.0001, 
    0.02, 
    T)


alpha = 1 - beta

alpha_bar = torch.cumprod(alpha, dim=0)
# (67)

alpha_bar_t = alpha_bar[ts]
print(alpha_bar_t.shape)

alpha_bar = alpha_bar[:, None, None, None]
# (67 x 1 x 1 x 1)



print('size of train_loader: ', len(train_loader))
print('shape of alpha: ', alpha.shape)
print('shape of alpha_bar: ', alpha_bar.shape)
print(alpha_bar[[0,1,2,3]].shape)
print('the shape of ts: ', ts.shape)
print(ts[[2, 0, 1, 3]])
print(ts)
print(alpha_bar[ts])

NameError: name 'batch' is not defined